# Achtung, die Kurve! - PPO Training (Colab)

Trains the CNN+PPO agent in `ai/` through the five curriculum phases (solo -> self-play -> multiplayer -> league -> items), fully headless (no rendering, pure NumPy/Pillow rasterization) so this runs at the environment's real simulation speed. Each phase is one `train.py` call; later phases warm-start from the previous phase's checkpoint with `--init-from`.

**Before running**: push your local repo (including the new `ai/` folder) to GitHub, then set `REPO_URL` below. Runtime -> Change runtime type -> GPU.

In [ ]:
!nvidia-smi

Also check how many CPU cores this runtime actually has - the environment simulation itself runs on CPU (only the PPO network updates use the GPU), and `n_envs` parallel worker processes competing for fewer cores than that will make the first rollout (before any log line prints) take a while. **If training looks silent for several minutes, that's normal on a CPU-constrained runtime - watch the progress bar rather than assuming it's frozen.** Set `N_ENVS` below to roughly match the core count.

In [ ]:
!nproc
N_ENVS = 4  # match this to the core count above; passed as --n-envs "$N_ENVS" to every phase call below

## 1. Setup

In [ ]:
REPO_URL = "https://github.com/jeremiassaur-2002/achtung-die-kurve_modified.git"
REPO_DIR = "/content/achtung-die-kurve"

import os
if not os.path.exists(REPO_DIR):
    !git clone "$REPO_URL" "$REPO_DIR"
%cd $REPO_DIR

In [ ]:
!pip install -q -r ai/requirements.txt

In [ ]:
# Mount Drive so checkpoints/logs survive Colab session resets - all runs write under ai/runs,
# so pointing --run-root at a Drive path is enough to persist everything across sessions.
from google.colab import drive
drive.mount("/content/drive")

RUN_ROOT = "/content/drive/MyDrive/achtung_kurve_runs"
import pathlib
pathlib.Path(RUN_ROOT).mkdir(parents=True, exist_ok=True)
BC_CKPT = f"{RUN_ROOT}/bc/bc_phase1.zip"  # optional Behavior-Cloning-Kickstart, siehe Abschnitt 2 unten

In [ ]:
%load_ext tensorboard
%tensorboard --logdir "$RUN_ROOT"

## 2. (Optional, empfohlen) Behavior-Cloning-Kickstart

PPO verbringt die ersten Millionen Schritte damit, per Zufall herauszufinden, dass Wand und eigene Spur toedlich sind - bei duennem Todes-Signal und einem lokal fast symmetrischen 3-Aktionen-Raum dauert das lange. Der regelbasierte Bot (`ai/core/env/rules_bot.py`, Schwierigkeit `hard`) beherrscht genau die Ueberlebensinstinkte, die Phase 1 lernen soll - `bc_pretrain.py` klont sein Verhalten supervised in ein PPO-Checkpoint, das Phase 1 unten automatisch per `--init-from` uebernimmt, falls es existiert.

Dauert nur wenige Minuten (deutlich kuerzer als das eigentliche PPO-Training) und ist komplett optional: **wird diese Zelle uebersprungen, startet Phase 1 unten stattdessen ganz normal bei Null.**

In [ ]:
!python -m ai.v1_0.training.bc_pretrain --config ai/v1_0/config/phase1.yaml --out "$BC_CKPT" --transitions 50000 --teacher hard

## 3. Phase 1 - solo survival
Wall + own-trail avoidance, no opponents, no items. Increase `--timesteps` for a real run (the config's own `total_timesteps` is used if you omit the flag). Startet automatisch vom Behavior-Cloning-Checkpoint aus Abschnitt 2, falls der dort erzeugt wurde - sonst normal bei Null.

Jeder Run legt unter seinem Ordner auf Drive `checkpoints/` (rotierend - es bleibt immer nur der neueste, und ein neuer wird erst vollstaendig geschrieben, bevor der alte geloescht wird) und `videos/` an (alle 500k Steps ein kurzes MP4 der aktuellen Policy, headless gerendert). Bricht die Colab-Session ab: **Zelle einfach nochmal ausfuehren** - `--resume auto` macht exakt am letzten Checkpoint weiter (inkl. Optimizer und Step-Zaehler).

In [ ]:
import os
init_from_arg = f'--init-from "{BC_CKPT}"' if os.path.exists(BC_CKPT) else ""
!python -m ai.v1_0.training.train --config ai/v1_0/config/phase1.yaml --run-root "$RUN_ROOT" --run-name phase1 --n-envs "$N_ENVS" --resume auto {init_from_arg}

## 4. Phase 2 - self-play (1 opponent, recent snapshots of self)

In [ ]:
PHASE1_CKPT = f"{RUN_ROOT}/phase1/final_model.zip"
!python -m ai.v1_0.training.train --config ai/v1_0/config/phase2.yaml --init-from "$PHASE1_CKPT" --run-root "$RUN_ROOT" --run-name phase2 --n-envs "$N_ENVS" --resume auto

## 5. Phase 3 - multiplayer curriculum (2..5 opponents)

In [ ]:
PHASE2_CKPT = f"{RUN_ROOT}/phase2/final_model.zip"
!python -m ai.v1_0.training.train --config ai/v1_0/config/phase3.yaml --init-from "$PHASE2_CKPT" --run-root "$RUN_ROOT" --run-name phase3 --n-envs "$N_ENVS" --resume auto

## 6. Phase 4 - league training (checkpoints + Elo)

In [ ]:
PHASE3_CKPT = f"{RUN_ROOT}/phase3/final_model.zip"
!python -m ai.v1_0.training.train --config ai/v1_0/config/phase4.yaml --init-from "$PHASE3_CKPT" --run-root "$RUN_ROOT" --run-name phase4 --n-envs "$N_ENVS" --resume auto

## 7. Phase 5 - items active

In [ ]:
PHASE4_CKPT = f"{RUN_ROOT}/phase4/final_model.zip"
!python -m ai.v1_0.training.train --config ai/v1_0/config/phase5.yaml --init-from "$PHASE4_CKPT" --run-root "$RUN_ROOT" --run-name phase5 --n-envs "$N_ENVS" --resume auto

## 8. Evaluate
Win rate / survival / placement / kills / item usage vs. random, every rule-based difficulty, and the league history.

In [ ]:
FINAL_CKPT = f"{RUN_ROOT}/phase5/final_model.zip"
LEAGUE_DIR = f"{RUN_ROOT}/phase5/league"
!python -m ai.core.evaluation.evaluate --checkpoint "$FINAL_CKPT" --league "$LEAGUE_DIR" --matches 30

## 9. Export weights
ONNX for `ai_bot.js` (in-browser inference via onnxruntime-web) + the native SB3 checkpoint + a plain PyTorch state_dict, then download them locally.

In [ ]:
EXPORT_DIR = f"{RUN_ROOT}/phase5/exported"
!python -m ai.core.export.export_onnx --checkpoint "$FINAL_CKPT" --out "$EXPORT_DIR/model.onnx"
!python -m ai.core.export.export_weights --checkpoint "$FINAL_CKPT" --out-dir "$EXPORT_DIR"

In [ ]:
# Download to your machine (files already persist on Drive at EXPORT_DIR regardless -
# use this if you want them locally right away instead of syncing Drive).
# model_data.js is the model bytes base64-embedded (export_onnx.py writes it automatically
# alongside model.onnx) - copy it next to model.onnx and index.html so the game works via
# file:// directly, no local server needed (onnxruntime-web can't fetch() local files there).
from google.colab import files
files.download(f"{EXPORT_DIR}/model.onnx")
files.download(f"{EXPORT_DIR}/model_data.js")
files.download(f"{EXPORT_DIR}/model_sb3.zip")
files.download(f"{EXPORT_DIR}/policy_state_dict.pt")

## Next steps

- Copy both `model.onnx` and `model_data.js` into the game's repo root, next to `index.html` - then just double-click `index.html` and tick a player's **KI** checkbox on the start screen (or `addAI('fred')` from the console). See `ai_bot.js`'s header comment for details.
- To keep training later, re-run any phase's cell with `--init-from` pointing at the latest checkpoint on Drive - everything under `RUN_ROOT` (metrics, league, self-play pool, reports) persists across sessions.